## Import libraries

In [83]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(override=True)

True

## Define directories

In [ ]:
notebook_dir = os.getcwd() 
project_root = os.path.dirname(notebook_dir)  
output_dir = os.path.join(project_root, 'data', 'processed') 
os.makedirs(output_dir, exist_ok=True)

## Load Data
### Tasks
- Load the patient, admissions, and icustays data from the CSV file specified in the environment variable.
- Display the columns of the Dataframes to verify that the data has been loaded correctly
- Display the shape of the Dataframes to see the number of rows and columns

In [ ]:
pat_data_path = os.getenv('PAT_DATA_PATH')
df_pat = pd.read_csv(pat_data_path)

In [ ]:
df_pat.columns

In [ ]:
df_pat.shape

In [ ]:
adm_data_path = os.getenv('ADM_DATA_PATH')
df_adm = pd.read_csv(adm_data_path)

In [ ]:
df_adm.columns

In [ ]:
df_adm.shape

In [ ]:
icustays_data_path = os.getenv('ICUSTAYS_DATA_PATH')
df_icustays = pd.read_csv(icustays_data_path)

In [ ]:
df_icustays.columns

In [ ]:
df_icustays.shape

## Stage 1: Build the Base ICU Stay Feature Table
### Tasks
- Create subsets for patients, admissions, and icustays extracting only the columns needed.
    Since we are comparing ICU patients, the ICU stay (stay_id) is chosen as the unit of analysis.
    Each row in the resulting dataset represents one ICU stat and serves as the foundation for all subsequent feature engineering.
- Merge subsets.
    Left join patients on subject_id then left join admissions on hadm_id. Left join ensures every ICU stay remains in the dataset.
- Update column names.
- Save stage 1 CSV into data/processed.

In [ ]:
patients_subset = df_pat[['subject_id', 'gender', 'anchor_age']]
admissions_subset = df_adm[['hadm_id','admission_type', 'race', 'hospital_expire_flag']]
icustays_subset = df_icustays[['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'los']]
 
df_stage1= pd.merge(icustays_subset, patients_subset, on='subject_id', how='left')
df_stage1 = pd.merge(df_stage1, admissions_subset, on='hadm_id', how='left')
df_stage1 = df_stage1.rename(columns={'anchor_age': 'age', 'los': 'icu_los_days'})

In [ ]:
output_file = os.path.join(output_dir, 'icu_stay_features_stage1.csv')
df_stage1.to_csv(output_file, index=False)
print(f"Saved directly to absolute root path:\n{output_file}")